In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("ptspark_assess_q1") \
    .getOrCreate()  
from pyspark.sql.types import StructType,StructField,IntegerType,StringType  
schema=StructType([StructField("customer",IntegerType(),True),
                   StructField("product_model",StringType(),True)  
])
data=(
    [1, "iphone13"],
 [1, "dell i5 core"],
 [2, "iphone13"],
 [2, "dell i5 core"],
 [3, "iphone13"],
 [3, "dell i5 core"],
 [1, "dell i3 core"],
 [1, "hp i5 core"],
 [1, "iphone14"],
 [3, "iphone14"],
 [4, "iphone13"]
)
df=spark.createDataFrame(data,schema=schema)
df.show()
from pyspark.sql.types import StructType,StructField,IntegerType,StringType  
schema=StructType([StructField("product_model",StringType(),True)
])
data=(
    ["iphone13"],
    ["dell i5 core"],
    ["dell i3 core"],
    ["hp i5 core"],
    ["iphone14"]

)
df2=spark.createDataFrame(data,schema=schema)
df2.show()

+--------+-------------+
|customer|product_model|
+--------+-------------+
|       1|     iphone13|
|       1| dell i5 core|
|       2|     iphone13|
|       2| dell i5 core|
|       3|     iphone13|
|       3| dell i5 core|
|       1| dell i3 core|
|       1|   hp i5 core|
|       1|     iphone14|
|       3|     iphone14|
|       4|     iphone13|
+--------+-------------+

+-------------+
|product_model|
+-------------+
|     iphone13|
| dell i5 core|
| dell i3 core|
|   hp i5 core|
|     iphone14|
+-------------+



In [0]:
from pyspark.sql.functions import col
df_iph=df.filter(df["product_model"]=="iphone13").alias("product_13")\
         
df_iph.show() 
df_iph14=df.filter(df["product_model"]=="iphone14") 
df_iph14.show()       

+--------+-------------+
|customer|product_model|
+--------+-------------+
|       1|     iphone13|
|       2|     iphone13|
|       3|     iphone13|
|       4|     iphone13|
+--------+-------------+

+--------+-------------+
|customer|product_model|
+--------+-------------+
|       1|     iphone14|
|       3|     iphone14|
+--------+-------------+



In [0]:
from pyspark.sql.functions import countDistinct,col
total_models = df.select("product_model").distinct().count()
customer_model_count = df.groupBy("customer") \
                         .agg(countDistinct("product_model").alias("model_count"))
customer_model_count.show()                         
result = customer_model_count.filter(col("model_count") == total_models)

result.select("customer").show()                        

+--------+-----------+
|customer|model_count|
+--------+-----------+
|       1|          5|
|       3|          3|
|       4|          1|
|       2|          2|
+--------+-----------+

+--------+
|customer|
+--------+
|       1|
+--------+



In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CreditCardMasking").getOrCreate()


data = [
    ("1234567891234567",),
    ("5678912345671234",),
    ("9123456712345678",),
    ("1234567812341122",),
    ("1234567812341342",)
]

columns = ["card_number"]

credit_card_df = spark.createDataFrame(data, columns)
credit_card_df.show()
print("number of partitions:", credit_card_df.rdd.getNumPartitions())
increased_df = credit_card_df.repartition(5)
print("Increased partitions:", increased_df.rdd.getNumPartitions())
original_df = increased_df.coalesce(credit_card_df.rdd.getNumPartitions())
print("original partitions:", original_df.rdd.getNumPartitions())


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def mask_card(card):
    return '*' * (len(card) - 4) + card[-4:]

mask_card_udf = udf(mask_card, StringType())

In [0]:
masked_df = credit_card_df.withColumn("masked_card_number", mask_card_udf(credit_card_df["card_number"]))
masked_df.show(truncate=False)

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Custom").getOrCreate()
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,TimestampType
custom=StructType([StructField("log_id",IntegerType(),True),
                   StructField("user$id",IntegerType(),True),
                   StructField("action",StringType(),True),
                   StructField("timestamp",StringType(),True)
])
Data=(
    [1, 101, 'login', '2023-09-05 08:30:00'],
    [2, 102, 'click', '2023-09-06 12:45:00'],
    [3, 101, 'click', '2023-09-07 14:15:00'],
    [4, 103, 'login', '2023-09-08 09:00:00'],
    [5, 102, 'logout', '2023-09-09 17:30:00'],
    [6, 101, 'click', '2023-09-10 11:20:00'],
    [7, 103, 'click', '2023-09-11 10:15:00'],
    [8, 102, 'click', '2023-09-12 13:10:00']
)
custom_data=spark.createDataFrame(Data,custom)
custom_data.show()
from pyspark.sql.functions import col
df_custom = custom_data.withColumn("event_time", col("timestamp").cast("timestamp")).drop("timestamp")
df_custom_rename=df_custom.withColumnRenamed("event_time","timestamp")

df_custom_rename.show()




+------+-------+------+-------------------+
|log_id|user$id|action|          timestamp|
+------+-------+------+-------------------+
|     1|    101| login|2023-09-05 08:30:00|
|     2|    102| click|2023-09-06 12:45:00|
|     3|    101| click|2023-09-07 14:15:00|
|     4|    103| login|2023-09-08 09:00:00|
|     5|    102|logout|2023-09-09 17:30:00|
|     6|    101| click|2023-09-10 11:20:00|
|     7|    103| click|2023-09-11 10:15:00|
|     8|    102| click|2023-09-12 13:10:00|
+------+-------+------+-------------------+

+------+-------+------+-------------------+
|log_id|user$id|action|          timestamp|
+------+-------+------+-------------------+
|     1|    101| login|2023-09-05 08:30:00|
|     2|    102| click|2023-09-06 12:45:00|
|     3|    101| click|2023-09-07 14:15:00|
|     4|    103| login|2023-09-08 09:00:00|
|     5|    102|logout|2023-09-09 17:30:00|
|     6|    101| click|2023-09-10 11:20:00|
|     7|    103| click|2023-09-11 10:15:00|
|     8|    102| click|2023-09-

In [0]:
from pyspark.sql.functions import count,countDistinct,col,aggregate,current_timestamp,expr, to_timestamp, lit,date_sub
last_7_days=date_sub(current_timestamp(), 7)
filtered_df = df_custom_rename.filter(col("timestamp")>= ("last_7_days"))
df_custom_filter=filtered_df .groupBy("user$id").agg(count(col("*")).alias("last_7_days"))
df_custom_filter.show()

+-------+-----------+
|user$id|last_7_days|
+-------+-----------+
+-------+-----------+



In [0]:
from pyspark.sql.functions import to_date
change_column=df_custom_rename.withColumn("login_date",to_date(col("timestamp")))
change_column.show()

+------+-------+------+-------------------+----------+
|log_id|user$id|action|          timestamp|login_date|
+------+-------+------+-------------------+----------+
|     1|    101| login|2023-09-05 08:30:00|2023-09-05|
|     2|    102| click|2023-09-06 12:45:00|2023-09-06|
|     3|    101| click|2023-09-07 14:15:00|2023-09-07|
|     4|    103| login|2023-09-08 09:00:00|2023-09-08|
|     5|    102|logout|2023-09-09 17:30:00|2023-09-09|
|     6|    101| click|2023-09-10 11:20:00|2023-09-10|
|     7|    103| click|2023-09-11 10:15:00|2023-09-11|
|     8|    102| click|2023-09-12 13:10:00|2023-09-12|
+------+-------+------+-------------------+----------+

